以上定义输出结构的四种模式中，我们都是通过调用“with_structured_output”来获取结构化输出结
果，除了这种方式外，还可以通过使用输出解释器来获取结构化输出结果。下面介绍这两种获取结构化
结果的方式。
4.1 使用with_structured_output
这种方式是
最新、
最简洁的API，直接让模型“理解”你需要的数据结构，并返回解析好的对象。
此外，我们可以在with_structured_output方法中传入
include_raw=True 参数，表示返回解析前的
始AIMessage ，从而访问令牌用量等元数据。

In [4]:
from dotenv import load_dotenv
import os
from langchain_deepseek import ChatDeepSeek

load_dotenv(override=True)
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = "https://api.deepseek.com"
model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}}  #关闭思考模式
)

In [5]:
from pydantic import BaseModel, Field
from rich import print as rprint


class Movie(BaseModel):
    """电影信息"""
    title: str = Field(description="电影标题")
    year: int = Field(description="上映年份")
    director: str = Field(description="导演")
    rating: float = Field(description="评分（10分制）")


# 设置模型结构化输出
model_with_structure = model.with_structured_output(Movie, include_raw=True)
# 调用模型并获取结构化输出
resp = model_with_structure.invoke("给我介绍下电影《星际穿越》")
print(type(resp))
rprint(resp)

<class 'dict'>


{
    'raw': AIMessage(
        content='',
        additional_kwargs={'refusal': None},
        response_metadata={
            'token_usage': {
                'completion_tokens': 36,
                'prompt_tokens': 351,
                'total_tokens': 387,
                'completion_tokens_details': None,
                'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256},
                'prompt_cache_hit_tokens': 256,
                'prompt_cache_miss_tokens': 95
            },
            'model_provider': 'deepseek',
            'model_name': 'deepseek-v4-flash',
            'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402',
            'id': '3f882e81-3999-4d34-aece-838809e6f254',
            'finish_reason': 'tool_calls',
            'logprobs': None
        },
        id='lc_run--019f608f-985b-7651-bec9-aaad9dbd542f-0',
        tool_calls=[
            {
                'name': 'Movie',
                'args': {'title': '星际穿越'},
                'id': 'call_00_U55iSNwGDH9iPEQQ6hra3806',
                'type': 'tool_call'
            }
        ],
        invalid_tool_calls=[],
        usage_metadata={
            'input_tokens': 351,
            'output_tokens': 36,
            'total_tokens': 387,
            'input_token_details': {'cache_read': 256},
            'output_token_details': {}
        }
    ),
    'parsing_error': 3 validation errors for Movie
year
  Field required [type=missing, input_value={'title': '星际穿越'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
director
  Field required [type=missing, input_value={'title': '星际穿越'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
rating
  Field required [type=missing, input_value={'title': '星际穿越'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing,
    'parsed': None
}

输出包含了完整的输出响应，包含三个字段
raw：返回的原始AIMessage。
parsed：解析后的输出
parsing_error：解析错误，当前用的是Pydantic，校验，格式不符合schema会导致报错。其它
三种方式不符合schema不会导致报错。

4.2
使用输出解析器(不推荐)
这种方法更传统，依赖于在提示词中明确指示模型输出特定格式的文本，然后使用解析器进行转换。
其流程是：提示词指导(引导生成指定类型）→ 模型生成文本 → 解析器转换。

In [7]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field


# 1. 创建提示词模板
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "回答用户问题,必须始终输出一个包含title(电影标题)和year(上映年份)的 JSON 对象"),
    ("human", "问题：{question}")
])

# 3. 定义结构


class Movie(BaseModel):
    """电影信息"""
    title: str = Field(description="电影标题")
    year: int = Field(description="上映年份")


# 4. 创建输出解析器
parser = JsonOutputParser(pydantic_object=Movie)
# 5. 创建链
chain = prompt_template | model | parser
# 6. 调用（返回字典）
response = chain.invoke({"question": "介绍电影《盗梦空间》"})
#
# response = parser.invoke(model.invoke(prompt_template.invoke({"question": "介绍电影《盗梦空间》"})))
print(response)

{'title': '盗梦空间', 'year': 2010}
